# Simulación de circuitos booleanos

Un circuito booleano es un DAG de compuertas que transforma $n$ bits de entrada en bits de salida.
Este cuaderno implementa un simulador de circuitos y lo usa para explorar la relación entre
tamaño, profundidad y la función computada.

**Artículo asociado:** [Circuitos booleanos y complejidad de circuitos](../../04-complejidad-computacional/07-circuitos-booleanos.md)

In [ ]:
class Circuit:
    """
    Circuito booleano representado como DAG.
    Cada nodo es una compuerta: 'AND', 'OR', 'NOT', 'INPUT', 'CONST'.
    """

    def __init__(self):
        self.nodes = {}   # id -> {'type': ..., 'inputs': [id, ...]}
        self.outputs = [] # lista de ids que son salidas
        self._next_id = 0

    def _new_id(self):
        nid = self._next_id
        self._next_id += 1
        return nid

    def input(self, name=None):
        """Añade un nodo de entrada y devuelve su id."""
        nid = self._new_id()
        self.nodes[nid] = {'type': 'INPUT', 'name': name or f'x{nid}', 'inputs': []}
        return nid

    def const(self, value):
        """Añade una constante 0 o 1."""
        nid = self._new_id()
        self.nodes[nid] = {'type': 'CONST', 'value': int(value), 'inputs': []}
        return nid

    def AND(self, a, b):
        nid = self._new_id()
        self.nodes[nid] = {'type': 'AND', 'inputs': [a, b]}
        return nid

    def OR(self, a, b):
        nid = self._new_id()
        self.nodes[nid] = {'type': 'OR', 'inputs': [a, b]}
        return nid

    def NOT(self, a):
        nid = self._new_id()
        self.nodes[nid] = {'type': 'NOT', 'inputs': [a]}
        return nid

    def XOR(self, a, b):
        """XOR = (a OR b) AND NOT(a AND b)."""
        return self.AND(self.OR(a, b), self.NOT(self.AND(a, b)))

    def mark_output(self, nid):
        self.outputs.append(nid)

    def evaluate(self, input_values):
        """
        Evalúa el circuito dado un dict {nombre_entrada: valor}.
        Devuelve los valores de los nodos de salida.
        """
        cache = {}

        def eval_node(nid):
            if nid in cache:
                return cache[nid]
            node = self.nodes[nid]
            if node['type'] == 'INPUT':
                result = int(input_values[node['name']])
            elif node['type'] == 'CONST':
                result = node['value']
            elif node['type'] == 'AND':
                result = eval_node(node['inputs'][0]) & eval_node(node['inputs'][1])
            elif node['type'] == 'OR':
                result = eval_node(node['inputs'][0]) | eval_node(node['inputs'][1])
            elif node['type'] == 'NOT':
                result = 1 - eval_node(node['inputs'][0])
            cache[nid] = result
            return result

        return [eval_node(out) for out in self.outputs]

    def size(self):
        """Número de compuertas (excluye entradas y constantes)."""
        return sum(1 for n in self.nodes.values() if n['type'] not in ('INPUT', 'CONST'))

    def depth(self):
        """Longitud del camino más largo desde entrada a salida."""
        memo = {}
        def dep(nid):
            if nid in memo:
                return memo[nid]
            node = self.nodes[nid]
            if node['type'] in ('INPUT', 'CONST'):
                d = 0
            else:
                d = 1 + max(dep(i) for i in node['inputs'])
            memo[nid] = d
            return d
        return max(dep(out) for out in self.outputs) if self.outputs else 0

print('Simulador de circuitos booleanos listo.')

## Ejemplo 1: semisumador

El semisumador computa la suma $s = a \oplus b$ y el acarreo $c = a \wedge b$.

In [ ]:
circ_semi = Circuit()
a = circ_semi.input('a')
b = circ_semi.input('b')
s = circ_semi.XOR(a, b)   # bit de suma
c = circ_semi.AND(a, b)   # bit de acarreo
circ_semi.mark_output(s)
circ_semi.mark_output(c)

print(f'Tamaño: {circ_semi.size()} compuertas')
print(f'Profundidad: {circ_semi.depth()}')
print()
print(f'{"a":>4} {"b":>4} | {"s (suma)":>8} {"c (acarreo)":>12}')
print('-' * 35)
for av in [0, 1]:
    for bv in [0, 1]:
        sv, cv = circ_semi.evaluate({'a': av, 'b': bv})
        print(f'{av:>4} {bv:>4} | {sv:>8} {cv:>12}')

## Ejemplo 2: sumador de 4 bits (ripple-carry)

Encadenamos 4 sumadores completos. Cada sumador completo recibe $a_i$, $b_i$ y el acarreo $c_{i-1}$,
y produce el bit de suma $s_i$ y el nuevo acarreo $c_i$.

Con sumadores encadenados la profundidad crece linealmente con el número de bits.

In [ ]:
def full_adder(circ, a, b, cin):
    """Devuelve (sum_bit, carry_out)."""
    ab  = circ.XOR(a, b)
    s   = circ.XOR(ab, cin)
    cout = circ.OR(circ.AND(a, b), circ.AND(ab, cin))
    return s, cout

def adder_n(n):
    """Sumador de n bits. Devuelve circuito y bits de salida [s_{n-1}, ..., s_0, carry_out]."""
    circ = Circuit()
    a_bits = [circ.input(f'a{i}') for i in range(n)]
    b_bits = [circ.input(f'b{i}') for i in range(n)]
    carry = circ.const(0)
    sum_bits = []
    for i in range(n):
        s, carry = full_adder(circ, a_bits[i], b_bits[i], carry)
        sum_bits.append(s)
    sum_bits.append(carry)
    for sb in sum_bits:
        circ.mark_output(sb)
    return circ, a_bits, b_bits, sum_bits

print('Parámetros del sumador ripple-carry:')
print(f'{"n (bits)":>10}  {"tamaño":>8}  {"profundidad":>12}')
for n in [1, 2, 4, 8, 16]:
    circ, _, _, _ = adder_n(n)
    print(f'{n:>10}  {circ.size():>8}  {circ.depth():>12}')

print()
print('Verificación del sumador de 4 bits:')
circ4, a4, b4, s4 = adder_n(4)
for av, bv in [(3, 5), (7, 8), (15, 1), (9, 6)]:
    vals = {f'a{i}': (av >> i) & 1 for i in range(4)}
    vals.update({f'b{i}': (bv >> i) & 1 for i in range(4)})
    result_bits = circ4.evaluate(vals)
    result = sum(b << i for i, b in enumerate(result_bits))
    ok = '✓' if result == av + bv else '✗'
    print(f'  {ok}  {av} + {bv} = {result}  (esperado {av + bv})')

## Ejemplo 3: construcción DNF

Cualquier función booleana $f: \{0,1\}^n \to \{0,1\}$ puede computarse con un circuito de
profundidad 3 usando su forma normal disyuntiva (DNF):

1. Para cada asignación donde $f = 1$, construir una cláusula AND de literales.
2. Combinar todas las cláusulas con un OR.

El tamaño es $O(n \cdot 2^n)$ en el peor caso.

In [ ]:
def build_dnf(n, truth_table):
    """
    Construye un circuito DNF para la función f cuyos valores están en truth_table
    (lista de longitud 2^n: truth_table[i] = f aplicada a la asignación i en binario).
    """
    circ = Circuit()
    inputs = [circ.input(f'x{i}') for i in range(n)]

    clauses = []
    for assignment, value in enumerate(truth_table):
        if not value:
            continue
        # Construir la cláusula AND para esta asignación
        literals = []
        for i in range(n):
            bit = (assignment >> i) & 1
            lit = inputs[i] if bit else circ.NOT(inputs[i])
            literals.append(lit)
        # AND de todos los literales
        clause = literals[0]
        for lit in literals[1:]:
            clause = circ.AND(clause, lit)
        clauses.append(clause)

    if not clauses:
        out = circ.const(0)
    elif len(clauses) == 1:
        out = clauses[0]
    else:
        out = clauses[0]
        for cl in clauses[1:]:
            out = circ.OR(out, cl)

    circ.mark_output(out)
    return circ

# Función de paridad de 3 bits: f(x0,x1,x2) = x0 XOR x1 XOR x2
parity3_table = [bin(i).count('1') % 2 for i in range(8)]
print('Tabla de verdad de la paridad de 3 bits:', parity3_table)

circ_par = build_dnf(3, parity3_table)
print(f'Tamaño DNF: {circ_par.size()} compuertas')
print(f'Profundidad DNF: {circ_par.depth()}')
print()
print('Verificación:')
for i in range(8):
    vals = {f'x{j}': (i >> j) & 1 for j in range(3)}
    resultado = circ_par.evaluate(vals)[0]
    esperado = parity3_table[i]
    ok = '✓' if resultado == esperado else '✗'
    bits = tuple((i >> j) & 1 for j in range(3))
    print(f'  {ok}  f{bits} = {resultado}  (esperado {esperado})')

## Ejemplo 4: tamaño mínimo vs. tamaño DNF

La construcción DNF da circuitos de profundidad 3 pero tamaño exponencial.
Para funciones simétricas como la paridad, existen circuitos mucho más pequeños.

In [ ]:
def build_xor_tree(circ, bits):
    """Construye un árbol XOR balanceado sobre una lista de bits."""
    if len(bits) == 1:
        return bits[0]
    mid = len(bits) // 2
    left = build_xor_tree(circ, bits[:mid])
    right = build_xor_tree(circ, bits[mid:])
    return circ.XOR(left, right)

import math

print('Comparación de tamaño y profundidad:')
print(f'{"n":>4}  {"DNF tamaño":>12}  {"XOR-árbol tamaño":>18}  {"XOR-árbol prof":>16}')
for n in range(2, 9):
    # DNF para paridad de n bits
    parity_table = [bin(i).count('1') % 2 for i in range(2**n)]
    circ_dnf = build_dnf(n, parity_table)

    # Árbol XOR
    circ_xor = Circuit()
    bits = [circ_xor.input(f'x{i}') for i in range(n)]
    out = build_xor_tree(circ_xor, bits)
    circ_xor.mark_output(out)

    print(f'{n:>4}  {circ_dnf.size():>12}  {circ_xor.size():>18}  {circ_xor.depth():>16}')

print()
print('El árbol XOR tiene tamaño O(n) y profundidad O(log n).')
print('La DNF tiene tamaño O(n·2^n) pero profundidad 3.')
print('Esto ilustra el compromiso tamaño-profundidad (size-depth tradeoff).')

## Ideas clave

- Cualquier función booleana tiene un circuito de profundidad 3 (DNF), pero con tamaño exponencial.
- Para funciones simétricas como la paridad, existen circuitos de tamaño $O(n)$ y profundidad $O(\log n)$ (árbol XOR).
- El teorema de Håstad demuestra que la paridad **no está en AC0**: ningún circuito de profundidad constante y tamaño polinomial puede computarla, aunque sí lo puede un circuito de profundidad $O(\log n)$.
- El sumador ripple-carry tiene profundidad $O(n)$; con carry-lookahead se baja a $O(\log n)$ con tamaño $O(n \log n)$.